# Discovering Cognitive Models with SPICE

**Tutorial Duration: ~1.5 hours**

In this tutorial, you will learn how to use **SPICE** (Sparse and Interpretable Cognitive Equations) to automatically discover reinforcement learning models from behavioral data.

SPICE fits **ensembles** of recurrent neural networks (RNNs) to behavioral data and then extracts concise mathematical equations describing the learned dynamics using sparse equation discovery (SINDy).

### What you will learn:
1. How to prepare data for SPICE (CSV format → SpiceDataset)
2. Implementing and fitting SPICE models of increasing complexity
3. Inspecting and interpreting discovered equations
4. Parameter recovery across different agent types

### Tutorial structure
We'll build up complexity step by step, running the **full pipeline** for each level:

| Level | Mechanism | Pipeline |
|-------|-----------|----------|
| Basic | Rescorla-Wagner learning | Data → Model → Fit → Inspect |
| + Forgetting | Value decay for unchosen actions | Data → Model → Fit → Inspect |
| + Choice Perseverance | Repeat bias | Data → Model → Fit → Inspect |
| Recovery | Mixed agents, structural differences | Generate → Fit → Analyze |

### Prerequisites
- Basic Python & PyTorch familiarity
- Understanding of reinforcement learning (Q-learning, Rescorla-Wagner)
- `pip install autospice`

---
## Setup

In [ ]:
# Uncomment if running on Google Colab:
# !pip uninstall -y numpy pandas
# !pip install numpy==1.26.4 pandas==2.2.2
# !pip install autospice

In [ ]:
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

np.random.seed(42)
torch.manual_seed(42)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {DEVICE}")

---
# Part 1: Data Format & The Bandit Task

## 1.1 The Two-Armed Bandit Task

In a two-armed bandit task, a participant repeatedly chooses between two options. Each option yields a **continuous reward** (between 0 and 1) drawn from a distribution centered on that option's mean reward. These means may drift over time, requiring the participant to continuously learn.

Let's first write a simulation of both the **environment** and a **Q-learning agent**.

In [ ]:
def simulate_bandit_qlearning(
    n_participants: int = 1,
    n_blocks: int = 5,
    n_trials: int = 100,
    n_actions: int = 2,
    # Agent parameters
    alpha_reward: float = 0.3,
    alpha_penalty: float = None,
    forget_rate: float = 0.0,
    alpha_choice: float = 0.0,
    beta_reward: float = 3.0,
    beta_choice: float = 0.0,
    # Environment parameters
    drift_rate: float = 0.1,
    reward_noise: float = 0.15,
    q_init: float = 0.5,
):
    """Simulate a Q-learning agent on a drifting two-armed bandit with continuous rewards.
    
    Each participant plays n_blocks independent blocks (new reward schedule per block).
    Rewards are drawn from N(mean_reward, reward_noise), clipped to [0, 1].
    
    Returns:
        df: DataFrame with columns [participant, block, choice, reward]
        trajectories: dict with internal agent states for visualization (indexed by session)
    """
    if alpha_penalty is None:
        alpha_penalty = alpha_reward
    
    records = []
    trajectories = {'Q': [], 'C': [], 'reward_probs': [], 'choices': [], 'rewards': []}
    
    for participant in range(n_participants):
        for block in range(n_blocks):
            reward_means = np.random.uniform(0.2, 0.8, size=n_actions)
            Q = np.full(n_actions, q_init)
            C = np.zeros(n_actions)
            
            session_Q, session_C, session_rp = [], [], []
            session_choices, session_rewards = [], []
            
            for trial in range(n_trials):
                session_Q.append(Q.copy())
                session_C.append(C.copy())
                session_rp.append(reward_means.copy())
                
                # Action selection (softmax)
                logits = beta_reward * Q + beta_choice * C
                logits -= logits.max()
                probs = np.exp(logits) / np.exp(logits).sum()
                choice = np.random.choice(n_actions, p=probs)
                
                # Get continuous reward: Gaussian around mean, clipped to [0, 1]
                reward = np.clip(np.random.normal(reward_means[choice], reward_noise), 0.0, 1.0)
                
                session_choices.append(choice)
                session_rewards.append(reward)
                records.append({
                    'participant': participant, 'block': block,
                    'choice': choice, 'reward': reward,
                })
                
                # Q-value update (Rescorla-Wagner)
                prediction_error = reward - Q[choice]
                alpha = alpha_reward if prediction_error >= 0 else alpha_penalty
                Q[choice] += alpha * prediction_error
                
                # Forgetting for unchosen actions
                for a in range(n_actions):
                    if a != choice:
                        Q[a] += forget_rate * (q_init - Q[a])
                
                # Choice perseverance
                C[choice] += alpha_choice * (1.0 - C[choice])
                for a in range(n_actions):
                    if a != choice:
                        C[a] += alpha_choice * (0.0 - C[a])
                
                # Drift environment
                reward_means += np.random.normal(0, drift_rate, size=n_actions)
                reward_means = np.clip(reward_means, 0.05, 0.95)
            
            trajectories['Q'].append(np.array(session_Q))
            trajectories['C'].append(np.array(session_C))
            trajectories['reward_probs'].append(np.array(session_rp))
            trajectories['choices'].append(np.array(session_choices))
            trajectories['rewards'].append(np.array(session_rewards))
    
    return pd.DataFrame(records), trajectories

## 1.2 Visualizing the task and agent behavior

In [ ]:
# Generate data from a simple RW agent
df_basic, traj_basic = simulate_bandit_qlearning(
    n_participants=1, n_blocks=5, n_trials=100,
    alpha_reward=0.3, beta_reward=3.0,
)

# Plot one example session
session_id = 0
fig, axes = plt.subplots(3, 1, figsize=(10, 7), sharex=True)

# Top: Reward means (drifting)
ax = axes[0]
rp = traj_basic['reward_probs'][session_id]
ax.plot(rp[:, 0], 'C0', alpha=0.7, label='Mean reward arm 0')
ax.plot(rp[:, 1], 'C1', alpha=0.7, label='Mean reward arm 1')
ax.set_ylabel('Mean reward')
ax.set_ylim(-0.05, 1.05)
ax.legend(loc='upper right')
ax.set_title('Bandit task: drifting reward means')

# Middle: Choices + received rewards (continuous)
ax = axes[1]
choices = traj_basic['choices'][session_id]
rewards = traj_basic['rewards'][session_id]
for t in range(len(choices)):
    color = 'C0' if choices[t] == 0 else 'C1'
    ax.scatter(t, rewards[t], color=color, s=12, alpha=0.5)
ax.set_ylabel('Reward received')
ax.set_ylim(-0.05, 1.05)
ax.set_title('Agent choices (color) and rewards received (continuous)')

# Bottom: Q-values
ax = axes[2]
Q = traj_basic['Q'][session_id]
ax.plot(Q[:, 0], 'C0', label='Q(arm 0)')
ax.plot(Q[:, 1], 'C1', label='Q(arm 1)')
ax.set_xlabel('Trial')
ax.set_ylabel('Q-value')
ax.legend(loc='upper right')
ax.set_title('Agent internal Q-values')

plt.tight_layout()
plt.show()

## 1.3 SPICE Data Format: `csv_to_dataset`

SPICE expects a DataFrame with at minimum: `participant`, `choice`, and optionally `reward`, `block`, `experiment`.

The `csv_to_dataset()` function converts this into a `SpiceDataset`:
- **One-hot encodes** choices and maps rewards to the chosen action column
- Applies a **time shift**: `xs[t]` = observation at trial $t$, `ys[t]` = action at $t+1$ (next-action prediction)
- Output shape: `(sessions, trials, within_trial_ts, features)`

In [ ]:
from spice import csv_to_dataset, SpiceDataset

dataset_basic = csv_to_dataset(
    file=df_basic,
    df_participant_id='participant',
    df_block='block',
    df_choice='choice',
    df_feedback='reward',
)

print(f"xs shape: {dataset_basic.xs.shape}  →  (sessions, trials, within_ts, features)")
print(f"ys shape: {dataset_basic.ys.shape}  →  (sessions, trials, within_ts, n_actions)")
print(f"")
print(f"Dataset attributes:")
print(f"  n_actions:         {dataset_basic.n_actions}")
print(f"  n_participants:    {dataset_basic.n_participants}")
print(f"  n_trials:          {dataset_basic.n_trials}")
print(f"  n_reward_features: {dataset_basic.n_reward_features}")
print()
print("Feature layout in xs:")
print("  [choice_0, choice_1, reward_0, reward_1, time_trial, trial, block, experiment, participant]")
print(f"  Example trial: {dataset_basic.xs[0, 3, 0, :].numpy()}")

---
# Part 2: Basic Rescorla-Wagner — Full Pipeline

The Rescorla-Wagner equation is:

$$V_{t+1} = V_t + \alpha \cdot (r_t - V_t) = (1-\alpha) \cdot V_t + \alpha \cdot r_t$$

We'll now discover this equation with SPICE by:
1. Generating data
2. Implementing the SPICE model
3. Fitting it
4. Inspecting the result

## 2.1 Data (already generated above)

We use `dataset_basic` from Part 1: 1 participant × 5 blocks × 100 trials, $\alpha=0.3$, $\beta=3$.

## 2.2 SPICE Model Implementation

A SPICE model has two parts:

### `SpiceConfig` — Defines the architecture
- **`library_setup`**: Maps each submodule to its control signals (inputs from data)
- **`memory_state`**: Named state variables with initial values

### Model class (subclass of `BaseModel`) — Implements the forward pass

For the basic RW model, we need:
- One submodule `reward_update_chosen` that receives the `reward` signal
- One memory state `value_reward` (the action values)

In [ ]:
from spice import SpiceConfig, BaseModel, SpiceEstimator

config_basic = SpiceConfig(
    library_setup={
        # Module name → list of control signals it receives from the data
        # The module's own state is ALWAYS included automatically in the SINDy library
        'reward_update_chosen': ['reward'],
    },
    memory_state={
        # State variable name → initial value
        'value_reward': 0.5,
    },
)

In [ ]:
class BasicRW(BaseModel):
    """Simplest SPICE model: learns the Rescorla-Wagner update for chosen actions."""
    
    def __init__(self, **kwargs):
        super().__init__(**kwargs)
        
        # Participant embeddings allow per-participant SINDy coefficients
        self.participant_embedding = self.setup_embedding(
            num_embeddings=self.n_participants
        )
        
        # Automatic module setup from config — registers all modules at once
        self.setup_modules_from_config()
        
        # Equivalent to manually calling:
        # self.setup_module(key_module='reward_update_chosen')  
        # Use setup_module() directly when you need custom settings like:
        #   self.setup_module(key_module='...', polynomial_degree=3, include_bias=False)
    
    def forward(self, inputs, prev_state=None):
        # Promote input to canonical 5D shape, extract signals
        spice_signals = self.init_forward_pass(inputs, prev_state)
        participant_embedding = self.participant_embedding(spice_signals.participant_ids)
        
        for timestep in spice_signals.trials:
            
            # Update value of the CHOSEN action based on reward
            self.call_module(
                key_module='reward_update_chosen',
                key_state='value_reward',
                action_mask=spice_signals.actions[timestep],  # 1 for chosen, 0 for unchosen
                inputs=spice_signals.feedback[timestep],      # reward signal
                participant_index=spice_signals.participant_ids,
                participant_embedding=participant_embedding,
            )
            
            # Logits = current state values → softmax → action probabilities
            spice_signals.logits[timestep] = self.state['value_reward']
        
        spice_signals = self.post_forward_pass(spice_signals)
        return spice_signals.logits, self.get_state()

### Key concepts:
- **`action_mask`**: One-hot tensor — only the chosen action's value gets updated; unchosen values are left unchanged
- **`key_state`**: The memory variable this module reads from and writes to
- **`inputs`**: Control signals (data features) fed to the RNN submodule
- **`spice_signals.logits`**: Raw scores for action selection (softmax applied during training loss)

## 2.3 Fitting with SpiceEstimator

### The Ensemble Concept

SPICE trains an **ensemble** of independent RNNs (each with their own weights) in parallel. This serves two purposes:

1. **Robust pruning**: A SINDy term is only kept if it is consistently non-zero across ensemble members (controlled by `sindy_ensemble_pruning`). This prevents overfitting to noise in a single RNN.

2. **Uncertainty quantification**: The spread of coefficients across ensemble members indicates how confident we are about each discovered term.

With `ensemble_size=10`, we train 10 independent RNNs simultaneously. A term must be significant in at least 70% of members (`sindy_ensemble_pruning=0.7`) to survive pruning.

In [ ]:
estimator_basic = SpiceEstimator(
    # Model
    spice_class=BasicRW,
    spice_config=config_basic,
    
    # Data properties
    n_actions=dataset_basic.n_actions,
    n_participants=dataset_basic.n_participants,
    
    # Training
    epochs=500,
    learning_rate=1e-2,
    ensemble_size=10,          # 10 independent RNNs for robust equation discovery
    
    # Sparsification
    sindy_weight=0.01,         # SINDy regularization strength
    sindy_threshold_pruning=0.05,   # minimum effect size to keep a term
    sindy_ensemble_pruning=0.7,     # required agreement across ensemble members
    
    verbose=True,
)

print("Fitting basic Rescorla-Wagner model...\n")
estimator_basic.fit(dataset_basic.xs, dataset_basic.ys)

## 2.4 Inspecting the Results

In [ ]:
# Print discovered equations
print("Discovered equation:")
print("-" * 50)
estimator_basic.print_spice_model(participant_id=0)
print()
print("Expected form (Rescorla-Wagner in logit space):")
print(f"  reward_update_chosen[t+1] = (1-alpha)*V[t] + alpha*beta*r[t]")
print(f"                            = {1-0.3:.1f}*V[t] + {0.3*3:.1f}*r[t]")

In [ ]:
# Visualize predictions vs ground truth for one session
predictions = estimator_basic.predict(dataset_basic.xs)
# predictions shape: (sessions, trials, within_ts, n_actions) — raw logits

session_id = 0
logits = predictions[session_id, :, 0, :]  # (trials, n_actions)
# Convert to action probabilities
action_probs = np.exp(logits) / np.exp(logits).sum(axis=-1, keepdims=True)

# Ground truth action probabilities from the agent's Q-values
Q_true = traj_basic['Q'][session_id]
logits_true = 3.0 * Q_true  # beta_reward=3.0
logits_true -= logits_true.max(axis=-1, keepdims=True)
probs_true = np.exp(logits_true) / np.exp(logits_true).sum(axis=-1, keepdims=True)

# SPICE predictions are shifted by 1 trial relative to ground truth:
# xs[t] = observation at trial t, so the model output at index t reflects
# the state *after* processing trial t (= agent state at t+1).
n = min(len(probs_true) - 1, len(action_probs))

fig, axes = plt.subplots(2, 1, figsize=(10, 5), sharex=True)

# Top: Action probabilities (SPICE vs ground truth)
ax = axes[0]
ax.plot(range(n), probs_true[1:n+1, 0], 'C0', alpha=0.4, label='True P(arm 0)')
ax.plot(range(n), probs_true[1:n+1, 1], 'C1', alpha=0.4, label='True P(arm 1)')
ax.plot(range(n), action_probs[:n, 0], 'C0--', label='SPICE P(arm 0)')
ax.plot(range(n), action_probs[:n, 1], 'C1--', label='SPICE P(arm 1)')
ax.set_ylabel('P(action)')
ax.set_ylim(-0.05, 1.05)
ax.legend(loc='upper right', fontsize=8)
ax.set_title('Action probabilities: ground truth vs SPICE')

# Bottom: Discovered values vs true Q-values
ax = axes[1]
ax.plot(range(n), Q_true[1:n+1, 0], 'C0', alpha=0.5, label='True Q(arm 0)')
ax.plot(range(n), Q_true[1:n+1, 1], 'C1', alpha=0.5, label='True Q(arm 1)')
ax.plot(range(n), logits[:n, 0] / 3.0, 'C0--', label='SPICE V(arm 0) / β')
ax.plot(range(n), logits[:n, 1] / 3.0, 'C1--', label='SPICE V(arm 1) / β')
ax.set_xlabel('Trial')
ax.set_ylabel('Value')
ax.legend(loc='upper right', fontsize=8)
ax.set_title('Learned values vs ground truth')

plt.tight_layout()
plt.show()

In [ ]:
# Programmatic access to coefficients
coefficients = estimator_basic.get_sindy_coefficients(aggregate=True)
terms = estimator_basic.get_candidate_terms()

print(f"Candidate terms: {terms['reward_update_chosen']}")
print(f"Coefficients:    {coefficients['reward_update_chosen'][0, 0].numpy().round(3)}")

---
# Part 3: Adding Forgetting — Full Pipeline

In real decision-making, people forget the values of options they haven't chosen recently. We model this with a second submodule that handles the **unchosen** actions.

Forgetting equation: $V^{\text{unchosen}}_{t+1} = (1-f) \cdot V^{\text{unchosen}}_t + f \cdot V_0$

## 3.1 Generate Data with Forgetting

In [ ]:
df_forget, traj_forget = simulate_bandit_qlearning(
    n_participants=1, n_blocks=5, n_trials=100,
    alpha_reward=0.3, beta_reward=3.0,
    forget_rate=0.1,   # <-- unchosen values drift back to 0.5
)

dataset_forget = csv_to_dataset(
    df_forget, df_participant_id='participant',
    df_block='block', df_choice='choice', df_feedback='reward',
)

# Visualize: compare Q-values with and without forgetting
fig, axes = plt.subplots(1, 2, figsize=(12, 3), sharey=True)
for ax, Q, title in zip(axes, [traj_basic['Q'][0], traj_forget['Q'][0]], 
                          ['No forgetting', 'With forgetting (f=0.1)']):
    ax.plot(Q[:, 0], 'C0', label='Q(arm 0)')
    ax.plot(Q[:, 1], 'C1', label='Q(arm 1)')
    ax.axhline(0.5, color='gray', linestyle=':', alpha=0.5)
    ax.set_xlabel('Trial')
    ax.set_ylabel('Q-value')
    ax.set_title(title)
    ax.legend()
plt.tight_layout()
plt.show()
print("Notice: with forgetting, unchosen values drift back toward 0.5 (gray line).")

## 3.2 Implement the Model

**Exercise:** Complete the config and model below.

The new module `reward_update_not_chosen`:
- Has **no control signals** (forgetting depends only on the current state)
- Updates the **unchosen** actions (`action_mask = 1 - actions`)
- Writes to the same `value_reward` state

In [ ]:
# EXERCISE: Complete the config

config_forget = SpiceConfig(
    library_setup={
        'reward_update_chosen': ['reward'],
        # TODO: Add the forgetting module (no control signals → empty list)
        # 'reward_update_not_chosen': ???,
    },
    memory_state={
        'value_reward': 0.5,
    },
)

In [ ]:
# EXERCISE: Complete the model

class ForgettingRW(BaseModel):
    """Rescorla-Wagner + forgetting for unchosen actions."""
    
    def __init__(self, **kwargs):
        super().__init__(**kwargs)
        self.participant_embedding = self.setup_embedding(num_embeddings=self.n_participants)
    
    def forward(self, inputs, prev_state=None):
        spice_signals = self.init_forward_pass(inputs, prev_state)
        participant_embedding = self.participant_embedding(spice_signals.participant_ids)
        
        for timestep in spice_signals.trials:
            # Update chosen action
            self.call_module(
                key_module='reward_update_chosen',
                key_state='value_reward',
                action_mask=spice_signals.actions[timestep],
                inputs=spice_signals.feedback[timestep],
                participant_index=spice_signals.participant_ids,
                participant_embedding=participant_embedding,
            )
            
            # TODO: Update unchosen actions (forgetting)
            # Hint: action_mask = 1 - spice_signals.actions[timestep], inputs = None
            # self.call_module(
            #     key_module=???,
            #     key_state=???,
            #     action_mask=???,
            #     inputs=???,
            #     participant_index=spice_signals.participant_ids,
            #     participant_embedding=participant_embedding,
            # )
            
            spice_signals.logits[timestep] = self.state['value_reward']
        
        spice_signals = self.post_forward_pass(spice_signals)
        return spice_signals.logits, self.get_state()

<details>
<summary><b>Click for solution</b></summary>

```python
config_forget = SpiceConfig(
    library_setup={
        'reward_update_chosen': ['reward'],
        'reward_update_not_chosen': [],  # No control signals
    },
    memory_state={'value_reward': 0.5},
)

class ForgettingRW(BaseModel):
    def __init__(self, **kwargs):
        super().__init__(**kwargs)
        self.participant_embedding = self.setup_embedding(num_embeddings=self.n_participants)
        self.setup_modules_from_config()
    
    def forward(self, inputs, prev_state=None):
        spice_signals = self.init_forward_pass(inputs, prev_state)
        participant_embedding = self.participant_embedding(spice_signals.participant_ids)
        
        for timestep in spice_signals.trials:
            self.call_module(
                key_module='reward_update_chosen',
                key_state='value_reward',
                action_mask=spice_signals.actions[timestep],
                inputs=spice_signals.feedback[timestep],
                participant_index=spice_signals.participant_ids,
                participant_embedding=participant_embedding,
            )
            self.call_module(
                key_module='reward_update_not_chosen',
                key_state='value_reward',
                action_mask=1 - spice_signals.actions[timestep],
                inputs=None,
                participant_index=spice_signals.participant_ids,
                participant_embedding=participant_embedding,
            )
            spice_signals.logits[timestep] = self.state['value_reward']
        
        spice_signals = self.post_forward_pass(spice_signals)
        return spice_signals.logits, self.get_state()
```
</details>

## 3.3 Fit & Inspect

In [ ]:
# Use the solution
config_forget = SpiceConfig(
    library_setup={'reward_update_chosen': ['reward'], 'reward_update_not_chosen': []},
    memory_state={'value_reward': 0.5},
)

class ForgettingRW(BaseModel):
    def __init__(self, **kwargs):
        super().__init__(**kwargs)
        self.participant_embedding = self.setup_embedding(num_embeddings=self.n_participants)
        self.setup_modules_from_config()
    
    def forward(self, inputs, prev_state=None):
        spice_signals = self.init_forward_pass(inputs, prev_state)
        participant_embedding = self.participant_embedding(spice_signals.participant_ids)
        for timestep in spice_signals.trials:
            self.call_module('reward_update_chosen', 'value_reward',
                action_mask=spice_signals.actions[timestep],
                inputs=spice_signals.feedback[timestep],
                participant_index=spice_signals.participant_ids,
                participant_embedding=participant_embedding)
            self.call_module('reward_update_not_chosen', 'value_reward',
                action_mask=1 - spice_signals.actions[timestep],
                inputs=None,
                participant_index=spice_signals.participant_ids,
                participant_embedding=participant_embedding)
            spice_signals.logits[timestep] = self.state['value_reward']
        spice_signals = self.post_forward_pass(spice_signals)
        return spice_signals.logits, self.get_state()

In [ ]:
estimator_forget = SpiceEstimator(
    # Model
    spice_class=ForgettingRW,
    spice_config=config_forget,

    # Data properties
    n_actions=dataset_forget.n_actions,
    n_participants=dataset_forget.n_participants,

    # Training
    epochs=500,
    learning_rate=1e-2,
    ensemble_size=10,

    # Sparsification
    sindy_weight=0.01,
    sindy_threshold_pruning=0.05,
    sindy_ensemble_pruning=0.7,

    verbose=True,
)
estimator_forget.fit(dataset_forget.xs, dataset_forget.ys)

In [ ]:
print("Discovered equations:")
print("-" * 60)
estimator_forget.print_spice_model()
print()
print("Expected:")
print(f"  reward_update_chosen[t+1]     ≈ 0.7*V[t] + 0.9*reward   (alpha=0.3, beta=3)")
print(f"  reward_update_not_chosen[t+1] ≈ 0.9*V[t] + constant     (1-f=0.9, f*V0*beta)")

In [ ]:
# Plot predictions vs ground truth
preds = estimator_forget.predict(dataset_forget.xs)
logits_f = preds[0, :, 0, :]
probs_f = np.exp(logits_f) / np.exp(logits_f).sum(axis=-1, keepdims=True)

# Ground truth action probabilities
Q_true = traj_forget['Q'][0]
logits_true = 3.0 * Q_true  # beta_reward=3.0
logits_true -= logits_true.max(axis=-1, keepdims=True)
probs_true = np.exp(logits_true) / np.exp(logits_true).sum(axis=-1, keepdims=True)

# Align: SPICE output at index t = agent state at t+1
n = min(len(probs_true) - 1, len(probs_f))

fig, axes = plt.subplots(2, 1, figsize=(10, 5), sharex=True)

# Top: Action probabilities
ax = axes[0]
ax.plot(range(n), probs_true[1:n+1, 0], 'C0', alpha=0.4, label='True P(arm 0)')
ax.plot(range(n), probs_true[1:n+1, 1], 'C1', alpha=0.4, label='True P(arm 1)')
ax.plot(range(n), probs_f[:n, 0], 'C0--', label='SPICE P(arm 0)')
ax.plot(range(n), probs_f[:n, 1], 'C1--', label='SPICE P(arm 1)')
ax.set_ylabel('P(action)')
ax.set_ylim(-0.05, 1.05)
ax.legend(loc='upper right', fontsize=8)
ax.set_title('Forgetting model \u2014 Action probabilities: ground truth vs SPICE')

# Bottom: Discovered values vs true Q-values
ax = axes[1]
ax.plot(range(n), Q_true[1:n+1, 0], 'C0', alpha=0.5, label='True Q(arm 0)')
ax.plot(range(n), Q_true[1:n+1, 1], 'C1', alpha=0.5, label='True Q(arm 1)')
ax.plot(range(n), logits_f[:n, 0] / 3.0, 'C0--', label='SPICE V(arm 0) / \u03b2')
ax.plot(range(n), logits_f[:n, 1] / 3.0, 'C1--', label='SPICE V(arm 1) / \u03b2')
ax.set_xlabel('Trial')
ax.set_ylabel('Value')
ax.legend(loc='upper right', fontsize=8)
ax.set_title('Learned values vs ground truth')

plt.tight_layout()
plt.show()

---
# Part 4: Adding Choice Perseverance — Full Pipeline

Choice perseverance captures the tendency to **repeat previous actions** regardless of reward outcomes. We model this as a separate memory state that contributes to the logits additively.

Here we use a single `choice_update` module that receives the **choice** as input and updates **all** items at once (no action_mask needed):

$$C_{t+1} = (1 - \alpha_c) \cdot C_t + \alpha_c \cdot \text{choice}_t$$

Since `choice` is one-hot encoded (e.g., `[1, 0]` for arm 0), this naturally increases the value for the chosen arm and leaves the unchosen at its current decayed value.

## 4.1 Generate Data

In [ ]:
df_full, traj_full = simulate_bandit_qlearning(
    n_participants=1, n_blocks=5, n_trials=100,
    alpha_reward=0.3, beta_reward=3.0,
    forget_rate=0.1,
    alpha_choice=0.5, beta_choice=1.5,  # <-- choice perseverance
)

dataset_full = csv_to_dataset(
    df_full, df_participant_id='participant',
    df_block='block', df_choice='choice', df_feedback='reward',
)

# Visualize the choice perseverance signal
fig, axes = plt.subplots(1, 2, figsize=(12, 3))

ax = axes[0]
Q = traj_full['Q'][0]
ax.plot(Q[:, 0], 'C0', label='Q(arm 0)')
ax.plot(Q[:, 1], 'C1', label='Q(arm 1)')
ax.set_title('Reward values (Q)')
ax.set_xlabel('Trial')
ax.legend()

ax = axes[1]
C = traj_full['C'][0]
ax.plot(C[:, 0], 'C0', label='C(arm 0)')
ax.plot(C[:, 1], 'C1', label='C(arm 1)')
ax.set_title('Choice perseverance values (C)')
ax.set_xlabel('Trial')
ax.legend()

plt.tight_layout()
plt.show()
print("Choice perseverance creates a 'momentum' — recently chosen actions get a bonus.")

## 4.2 Implement the Model

**Exercise:** Add the `choice_update` module.

Key differences from the reward modules:
- `action_mask=None` — updates all items simultaneously
- `inputs=spice_signals.actions[timestep]` — receives the choice as a control signal
- Writes to a separate state: `key_state='value_choice'`

In [ ]:
# EXERCISE: Complete the config and model

config_full = SpiceConfig(
    library_setup={
        'reward_update_chosen': ['reward'],
        'reward_update_not_chosen': [],
        # TODO: Add choice module. What control signal does it need?
        # Hint: it receives 'choice' as input
        # 'choice_update': ???,
    },
    memory_state={
        'value_reward': 0.5,
        # TODO: Add choice state (initial value 0)
        # 'value_choice': ???,
    },
)

class FullRW(BaseModel):
    """Rescorla-Wagner + Forgetting + Choice Perseverance."""
    
    def __init__(self, **kwargs):
        super().__init__(**kwargs)
        self.participant_embedding = self.setup_embedding(num_embeddings=self.n_participants)
    
    def forward(self, inputs, prev_state=None):
        spice_signals = self.init_forward_pass(inputs, prev_state)
        participant_embedding = self.participant_embedding(spice_signals.participant_ids)
        
        for timestep in spice_signals.trials:
            # Reward updates (same as before)
            self.call_module('reward_update_chosen', 'value_reward',
                action_mask=spice_signals.actions[timestep],
                inputs=spice_signals.feedback[timestep],
                participant_index=spice_signals.participant_ids,
                participant_embedding=participant_embedding)
            self.call_module('reward_update_not_chosen', 'value_reward',
                action_mask=1 - spice_signals.actions[timestep],
                inputs=None,
                participant_index=spice_signals.participant_ids,
                participant_embedding=participant_embedding)
            
            # TODO: Choice perseverance update
            # self.call_module('choice_update', 'value_choice',
            #     action_mask=???,    # updates all items at once
            #     inputs=???,         # what signal drives choice perseverance?
            #     participant_index=spice_signals.participant_ids,
            #     participant_embedding=participant_embedding)
            
            # Logits = sum of reward and choice values
            spice_signals.logits[timestep] = self.state['value_reward'] + self.state['value_choice']
        
        spice_signals = self.post_forward_pass(spice_signals)
        return spice_signals.logits, self.get_state()

<details>
<summary><b>Click for solution</b></summary>

```python
config_full = SpiceConfig(
    library_setup={
        'reward_update_chosen': ['reward'],
        'reward_update_not_chosen': [],
        'choice_update': ['choice'],  # receives choice as control signal
    },
    memory_state={
        'value_reward': 0.5,
        'value_choice': 0.0,
    },
)

# In the forward pass:
self.call_module('choice_update', 'value_choice',
    action_mask=None,                          # update ALL items
    inputs=spice_signals.actions[timestep],    # choice signal (one-hot)
    participant_index=spice_signals.participant_ids,
    participant_embedding=participant_embedding)
```
</details>

## 4.3 Fit & Inspect

In [ ]:
# Full solution
config_full = SpiceConfig(
    library_setup={
        'reward_update_chosen': ['reward'],
        'reward_update_not_chosen': [],
        'choice_update': ['choice'],
    },
    memory_state={'value_reward': 0.5, 'value_choice': 0.0},
)

class FullRW(BaseModel):
    def __init__(self, **kwargs):
        super().__init__(**kwargs)
        self.participant_embedding = self.setup_embedding(num_embeddings=self.n_participants)
        self.setup_modules_from_config()
    
    def forward(self, inputs, prev_state=None):
        spice_signals = self.init_forward_pass(inputs, prev_state)
        participant_embedding = self.participant_embedding(spice_signals.participant_ids)
        for timestep in spice_signals.trials:
            self.call_module('reward_update_chosen', 'value_reward',
                action_mask=spice_signals.actions[timestep],
                inputs=spice_signals.feedback[timestep],
                participant_index=spice_signals.participant_ids,
                participant_embedding=participant_embedding)
            self.call_module('reward_update_not_chosen', 'value_reward',
                action_mask=1 - spice_signals.actions[timestep],
                inputs=None,
                participant_index=spice_signals.participant_ids,
                participant_embedding=participant_embedding)
            self.call_module('choice_update', 'value_choice',
                action_mask=None,
                inputs=spice_signals.actions[timestep],
                participant_index=spice_signals.participant_ids,
                participant_embedding=participant_embedding)
            spice_signals.logits[timestep] = self.state['value_reward'] + self.state['value_choice']
        spice_signals = self.post_forward_pass(spice_signals)
        return spice_signals.logits, self.get_state()

In [ ]:
estimator_full = SpiceEstimator(
    # Model
    spice_class=FullRW,
    spice_config=config_full,

    # Data properties
    n_actions=dataset_full.n_actions,
    n_participants=dataset_full.n_participants,

    # Training
    epochs=700,
    learning_rate=1e-2,
    ensemble_size=10,

    # Sparsification
    sindy_weight=0.01,
    sindy_threshold_pruning=0.05,
    sindy_ensemble_pruning=0.7,

    verbose=True,
)

print("Fitting full model...")
estimator_full.fit(dataset_full.xs, dataset_full.ys)

In [ ]:
print("Discovered equations:")
print("-" * 60)
estimator_full.print_spice_model()
print()
print("Expected structure:")
print("  reward_update_chosen[t+1]     ~ (1-alpha)*V[t] + alpha*beta_r*reward")
print("  reward_update_not_chosen[t+1] ~ (1-f)*V[t] + f*V0*beta_r")
print("  choice_update[t+1]            ~ (1-alpha_c)*C[t] + alpha_c*beta_c*choice")

### Note: Redundant polynomial terms with binary inputs

You may notice that the `choice_update` equation during training contains both `choice` and `choice^2` terms:

```
choice_update[t+1] = ... + 0.177 choice + 0.177 choice^2
```

Since `choice` is binary (0 or 1), `choice == choice^2` — squaring a binary variable doesn't change it. These terms are **mathematically identical**, so their coefficients can simply be summed: `0.177 + 0.177 = 0.354 choice`. SPICE distributes the coefficient arbitrarily across the redundant terms.

**How to prevent this:**
1. **Set `polynomial_degree=1`** for the choice module: use `setup_module(key_module='choice_update', polynomial_degree=1)` instead of `setup_modules_from_config()`, which eliminates the `choice^2` term entirely
2. **Mask specific terms** in the SINDy coefficient prior: set entries in `model.sindy_coefficients_presence['choice_update']` to zero for unwanted terms before training
3. **Build a custom candidate library** that excludes squared terms for binary inputs

This is a general consideration when using polynomial SINDy libraries: terms that are algebraically identical for your input domain introduce redundancy. For continuous-valued inputs (like `reward`), `reward^2` is a genuinely different term — but for binary flags, higher-order powers are redundant.

In [ ]:
# Plot predictions vs ground truth
preds = estimator_full.predict(dataset_full.xs)
logits_full = preds[0, :, 0, :]
probs_full = np.exp(logits_full) / np.exp(logits_full).sum(axis=-1, keepdims=True)

# Ground truth action probabilities (includes choice perseverance)
Q_true = traj_full['Q'][0]
C_true = traj_full['C'][0]
logits_true = 3.0 * Q_true + 1.5 * C_true  # beta_reward=3.0, beta_choice=1.5
logits_true -= logits_true.max(axis=-1, keepdims=True)
probs_true = np.exp(logits_true) / np.exp(logits_true).sum(axis=-1, keepdims=True)

# Align: SPICE output at index t = agent state at t+1
n = min(len(probs_true) - 1, len(probs_full))

fig, axes = plt.subplots(2, 1, figsize=(10, 5), sharex=True)

# Top: Action probabilities
ax = axes[0]
ax.plot(range(n), probs_true[1:n+1, 0], 'C0', alpha=0.4, label='True P(arm 0)')
ax.plot(range(n), probs_true[1:n+1, 1], 'C1', alpha=0.4, label='True P(arm 1)')
ax.plot(range(n), probs_full[:n, 0], 'C0--', label='SPICE P(arm 0)')
ax.plot(range(n), probs_full[:n, 1], 'C1--', label='SPICE P(arm 1)')
ax.set_ylabel('P(action)')
ax.set_ylim(-0.05, 1.05)
ax.legend(loc='upper right', fontsize=8)
ax.set_title('Full model \u2014 Action probabilities: ground truth vs SPICE')

# Bottom: Combined logits (reward + choice values)
ax = axes[1]
logits_true_raw = 3.0 * Q_true + 1.5 * C_true  # before centering
ax.plot(range(n), logits_true_raw[1:n+1, 0], 'C0', alpha=0.5, label='True logit(arm 0)')
ax.plot(range(n), logits_true_raw[1:n+1, 1], 'C1', alpha=0.5, label='True logit(arm 1)')
ax.plot(range(n), logits_full[:n, 0], 'C0--', label='SPICE logit(arm 0)')
ax.plot(range(n), logits_full[:n, 1], 'C1--', label='SPICE logit(arm 1)')
ax.set_xlabel('Trial')
ax.set_ylabel('Logit (\u03b2\u2084Q + \u03b2\u1d9cC)')
ax.legend(loc='upper right', fontsize=8)
ax.set_title('Combined logits: ground truth vs SPICE')

plt.tight_layout()
plt.show()

---
# Part 5: Applying SPICE to Real Clinical Data

So far, we've tested SPICE on synthetic data where the ground truth is known. Now we apply it to **real behavioral data** from a clinical study.

### Dataset: Dezfouli et al. (2019)
- **Task**: Two-armed bandit with drifting reward probabilities
- **101 participants** across 3 diagnostic groups: Control, Depression, Bipolar
- **12 blocks** per participant, ~109 trials per block, binary rewards (0/1)

### Key question
Can SPICE discover **structural differences** in reinforcement learning mechanisms between diagnostic groups — not just different parameter values, but fundamentally different equation structures?

### What we'll do
1. Load and prepare the clinical data with `additional_inputs` for diagnosis metadata
2. Design an **extended model** with derived features (the fun part!)
3. Fit SPICE and evaluate model quality
4. Analyze coefficient distributions and structural group differences
5. Generate publication-quality figures

In [ ]:
import seaborn as sns
from scipy import stats
from sklearn.linear_model import LogisticRegression

# Load clinical dataset
df_clinical = pd.read_csv('../weinhardt2026/studies/dezfouli2019/data/dezfouli2019.csv')

print(f"Dataset shape: {df_clinical.shape}")
print(f"Columns: {df_clinical.columns.tolist()}")
print(f"\nParticipants per diagnostic group:")
print(df_clinical.groupby('diag')['participant'].nunique())
print(f"\nReward type: binary (0/1), values = {sorted(df_clinical['reward'].unique())}")
print(f"Trials per block: ~{df_clinical.groupby(['participant', 'block']).size().median():.0f}")
df_clinical.head()

In [ ]:
# Build diagnosis lookup BEFORE csv_to_dataset remaps participant IDs to 0-indexed integers
unique_participants = df_clinical['participant'].unique()
participant_diag = df_clinical.groupby('participant')['diag'].first()
diag_lookup = {i: participant_diag[pid] for i, pid in enumerate(unique_participants)}

print(f"Diagnosis mapping (first 5): {dict(list(diag_lookup.items())[:5])}")
print(f"Group sizes: { {g: sum(1 for v in diag_lookup.values() if v == g) for g in ['Control', 'Depression', 'Bipolar']} }")

# Create dataset with 'diag' as additional input
config_clinical = SpiceConfig(
    library_setup={
        'reward_update_chosen': ['reward'],
        'reward_update_not_chosen': [],
        'choice_update': ['choice'],
    },
    memory_state={'value_reward': 0.5, 'value_choice': 0.0},
    additional_inputs=('diag',),
)

dataset_clinical = csv_to_dataset(
    file=df_clinical,
    df_participant_id='participant',
    df_block='block',
    df_choice='choice',
    df_feedback='reward',
    additional_inputs=['diag'],
)
dataset_clinical.normalize_rewards()

print(f"\nDataset: {dataset_clinical.n_participants} participants, "
      f"{dataset_clinical.xs.shape[0]} sessions, "
      f"{dataset_clinical.xs.shape[1]} trials")

## 5.1 Designing a More Expressive Model

A key strength of SPICE is that you can **compute derived features** in the model's `forward()` method and feed them as additional control signals to SINDy modules. This is where the fun begins!

Examples of useful derived features:
- **Prediction error**: `reward - value` — the surprise signal that drives learning
- **Reward history**: running average of recent rewards
- **Choice switching**: did the participant switch actions?
- **Trial counter**: captures time-dependent effects

These precomputed features make the model more **polynomial-amenable** — a core SPICE design principle. Instead of requiring the RNN to learn complex nonlinear interactions internally (which SINDy can't easily approximate), we externalize them as explicit inputs.

The `additional_inputs` mechanism (like `diag` above) lets us pass metadata through the dataset without including it in the SINDy library — useful for group-level analysis later.

In [ ]:
# EXERCISE: Extend the FullRW model with derived features
# 
# Ideas to try:
# - Compute prediction error: reward - current value
# - Track reward history (running average)
# - Compute choice switching signal
# - Add prediction error as a control signal to a module
#
# Start with the basic structure and add your own derived features!

class ClinicalRW(BaseModel):
    """Extended RL model with derived features for clinical data analysis."""
    
    def __init__(self, **kwargs):
        super().__init__(**kwargs)
        self.participant_embedding = self.setup_embedding(num_embeddings=self.n_participants)
        # TODO: call setup_modules_from_config() or set up modules manually
    
    def forward(self, inputs, prev_state=None):
        spice_signals = self.init_forward_pass(inputs, prev_state)
        participant_embedding = self.participant_embedding(spice_signals.participant_ids)
        
        for timestep in spice_signals.trials:
            # TODO: Compute derived features here
            # Example: prediction_error = spice_signals.feedback[timestep] - self.state['value_reward']
            
            # TODO: Call modules (reward_update_chosen, reward_update_not_chosen, choice_update)
            # Hint: same pattern as FullRW from Part 4
            
            # TODO: Set logits = value_reward + value_choice
            pass
        
        spice_signals = self.post_forward_pass(spice_signals)
        return spice_signals.logits, self.get_state()

<details>
<summary><b>Click for solution</b></summary>

```python
class ClinicalRW(BaseModel):
    def __init__(self, **kwargs):
        super().__init__(**kwargs)
        self.participant_embedding = self.setup_embedding(num_embeddings=self.n_participants)
        self.setup_modules_from_config()
    
    def forward(self, inputs, prev_state=None):
        spice_signals = self.init_forward_pass(inputs, prev_state)
        participant_embedding = self.participant_embedding(spice_signals.participant_ids)
        
        for timestep in spice_signals.trials:
            self.call_module('reward_update_chosen', 'value_reward',
                action_mask=spice_signals.actions[timestep],
                inputs=spice_signals.feedback[timestep],
                participant_index=spice_signals.participant_ids,
                participant_embedding=participant_embedding)
            self.call_module('reward_update_not_chosen', 'value_reward',
                action_mask=1 - spice_signals.actions[timestep],
                inputs=None,
                participant_index=spice_signals.participant_ids,
                participant_embedding=participant_embedding)
            self.call_module('choice_update', 'value_choice',
                action_mask=None,
                inputs=spice_signals.actions[timestep],
                participant_index=spice_signals.participant_ids,
                participant_embedding=participant_embedding)
            spice_signals.logits[timestep] = self.state['value_reward'] + self.state['value_choice']
        
        spice_signals = self.post_forward_pass(spice_signals)
        return spice_signals.logits, self.get_state()
```
</details>

In [ ]:
# Use the solution
class ClinicalRW(BaseModel):
    def __init__(self, **kwargs):
        super().__init__(**kwargs)
        self.participant_embedding = self.setup_embedding(num_embeddings=self.n_participants)
        self.setup_modules_from_config()
    
    def forward(self, inputs, prev_state=None):
        spice_signals = self.init_forward_pass(inputs, prev_state)
        participant_embedding = self.participant_embedding(spice_signals.participant_ids)
        for timestep in spice_signals.trials:
            self.call_module('reward_update_chosen', 'value_reward',
                action_mask=spice_signals.actions[timestep],
                inputs=spice_signals.feedback[timestep],
                participant_index=spice_signals.participant_ids,
                participant_embedding=participant_embedding)
            self.call_module('reward_update_not_chosen', 'value_reward',
                action_mask=1 - spice_signals.actions[timestep],
                inputs=None,
                participant_index=spice_signals.participant_ids,
                participant_embedding=participant_embedding)
            self.call_module('choice_update', 'value_choice',
                action_mask=None,
                inputs=spice_signals.actions[timestep],
                participant_index=spice_signals.participant_ids,
                participant_embedding=participant_embedding)
            spice_signals.logits[timestep] = self.state['value_reward'] + self.state['value_choice']
        spice_signals = self.post_forward_pass(spice_signals)
        return spice_signals.logits, self.get_state()

print("ClinicalRW model defined.")

In [ ]:
estimator_clinical = SpiceEstimator(
    # Model
    spice_class=ClinicalRW,
    spice_config=config_clinical,

    # Data properties
    n_actions=dataset_clinical.n_actions,
    n_participants=dataset_clinical.n_participants,

    # Training
    epochs=700,
    warmup_steps=350,
    learning_rate=1e-2,
    ensemble_size=10,

    # Sparsification
    sindy_weight=0.01,
    sindy_threshold_pruning=0.05,
    sindy_ensemble_pruning=0.7,

    verbose=True,
)

print(f"Fitting SPICE to clinical data ({dataset_clinical.n_participants} participants)...")
estimator_clinical.fit(dataset_clinical.xs, dataset_clinical.ys)

## 5.2 Model Evaluation

We evaluate model quality by comparing predictions against actual participant choices using **trial-level log-likelihood**. We compare two modes:
- **SPICE-RNN**: The full RNN predictions (before equation extraction)
- **SPICE-EQ**: The sparse equation predictions (after SINDy)

If the equations capture the essential dynamics, SPICE-EQ performance should be close to SPICE-RNN.

In [ ]:
def evaluate_model(estimator, dataset, use_sindy_mode):
    """Compute NLL, trial likelihood, and BIC for a SPICE model."""
    estimator.model.eval()
    estimator.model.use_sindy = use_sindy_mode
    
    with torch.no_grad():
        logits, _ = estimator.model(dataset.xs)
    
    # Convert to choice probabilities
    probs = torch.softmax(logits, dim=-1)
    targets = dataset.ys
    
    # Mask NaN-padded trials
    mask = ~torch.isnan(targets[..., 0])
    probs_valid = probs[mask]
    targets_valid = targets[mask]
    
    # Log-likelihood
    epsilon = 1e-9
    ll = (targets_valid * torch.log(probs_valid.clamp(min=epsilon))).sum(dim=-1)
    nll = -ll.sum().item()
    trial_ll = ll.mean().item()
    
    # BIC
    n_params = estimator.count_sindy_coefficients().mean().item() if use_sindy_mode else sum(
        p.numel() for p in estimator.model.parameters())
    n_samples = mask.sum().item()
    bic = 2 * nll + n_params * np.log(n_samples)
    
    return {
        'NLL': nll,
        'Trial LL': trial_ll,
        'Trial Likelihood': np.exp(trial_ll),
        'BIC': bic,
        'n_params': n_params,
    }

scores_rnn = evaluate_model(estimator_clinical, dataset_clinical, use_sindy_mode=False)
scores_eq = evaluate_model(estimator_clinical, dataset_clinical, use_sindy_mode=True)

print(f"{'Metric':<20} {'SPICE-RNN':>12} {'SPICE-EQ':>12}")
print("-" * 46)
for key in ['NLL', 'Trial Likelihood', 'BIC', 'n_params']:
    fmt = '.2f' if key != 'n_params' else '.0f'
    print(f"{key:<20} {scores_rnn[key]:>12{fmt}} {scores_eq[key]:>12{fmt}}")

In [ ]:
# Group participants by diagnosis
control_ids = [pid for pid, diag in diag_lookup.items() if diag == 'Control']
depression_ids = [pid for pid, diag in diag_lookup.items() if diag == 'Depression']
bipolar_ids = [pid for pid, diag in diag_lookup.items() if diag == 'Bipolar']

print(f"Control ({len(control_ids)} participants):")
estimator_clinical.print_spice_model(participant_id=control_ids[0])
print()
print(f"Depression ({len(depression_ids)} participants):")
estimator_clinical.print_spice_model(participant_id=depression_ids[0])
print()
print(f"Bipolar ({len(bipolar_ids)} participants):")
estimator_clinical.print_spice_model(participant_id=bipolar_ids[0])

## 5.3 Coefficient Distribution Analysis

We now analyze how the discovered SINDy coefficients distribute across the 101 participants. Key questions:
- Which terms are universally present vs. participant-specific?
- How do coefficient values vary across the population?
- Are there visible clusters aligned with diagnostic groups?

In [ ]:
# Extract coefficients
coefficients = estimator_clinical.get_sindy_coefficients(aggregate=True)
candidate_terms = estimator_clinical.get_candidate_terms()
modules = estimator_clinical.get_modules()

# --- Presence rates per module ---
fig, axes = plt.subplots(1, len(modules), figsize=(5 * len(modules), 4), sharey=True)
if len(modules) == 1:
    axes = [axes]

for ax, module in zip(axes, modules):
    coefs = coefficients[module][:, 0].numpy()  # (P, terms)
    terms = candidate_terms[module]
    presence = (np.abs(coefs) > 1e-6).mean(axis=0)
    
    bars = ax.bar(range(len(terms)), presence, color='steelblue', alpha=0.8)
    ax.set_xticks(range(len(terms)))
    ax.set_xticklabels(terms, rotation=45, ha='right', fontsize=8)
    ax.set_ylabel('Presence rate')
    ax.set_title(module)
    ax.set_ylim(0, 1.05)
    ax.axhline(0.5, color='gray', linestyle=':', alpha=0.5)

plt.suptitle('Coefficient presence rates across participants', fontsize=12)
plt.tight_layout()
plt.show()

# --- Sparsity heatmap grouped by diagnosis ---
# Build full coefficient matrix: participants x all terms
all_terms = []
coef_matrix = []
for module in modules:
    terms = candidate_terms[module]
    coefs = coefficients[module][:, 0].numpy()  # (P, terms)
    for t_idx, term in enumerate(terms):
        all_terms.append(f"{module}\n{term}")
        coef_matrix.append(coefs[:, t_idx])

coef_matrix = np.array(coef_matrix).T  # (P, all_terms)

# Sort participants by diagnostic group
group_order = ['Control', 'Depression', 'Bipolar']
sorted_indices = []
group_boundaries = []
for grp in group_order:
    grp_ids = [pid for pid, d in diag_lookup.items() if d == grp]
    sorted_indices.extend(grp_ids)
    group_boundaries.append(len(sorted_indices))

coef_sorted = coef_matrix[sorted_indices]
vmax = max(abs(coef_sorted.min()), abs(coef_sorted.max()))

fig, ax = plt.subplots(figsize=(max(10, len(all_terms) * 0.8), 8))
im = ax.imshow(coef_sorted, aspect='auto', cmap='RdBu_r', vmin=-vmax, vmax=vmax,
               interpolation='nearest')

# Group separators
for b in group_boundaries[:-1]:
    ax.axhline(b - 0.5, color='black', linewidth=2)

# Group labels
prev = 0
for grp, b in zip(group_order, group_boundaries):
    mid = (prev + b) / 2
    ax.text(-0.5, mid, grp, ha='right', va='center', fontsize=9, fontweight='bold',
            transform=ax.get_yaxis_transform())
    prev = b

ax.set_xticks(range(len(all_terms)))
ax.set_xticklabels(all_terms, rotation=45, ha='right', fontsize=7)
ax.set_ylabel('Participant (sorted by diagnosis)')
ax.set_title('Equation fingerprint: SINDy coefficients per participant')
plt.colorbar(im, ax=ax, label='Coefficient value', shrink=0.8)
plt.tight_layout()
plt.show()

## 5.4 Structural Group Differences

Do diagnostic groups differ in which equation terms are **present** (active vs. pruned to zero)?

We test this using **logistic regression**: for each SINDy coefficient, we predict its presence (1/0) from group membership. The regression coefficient (beta) quantifies the log-odds ratio — how much more/less likely the term is to be present in one group vs. the reference group (Control).

In [ ]:
# Build presence DataFrame
records = []
for p_idx in range(dataset_clinical.n_participants):
    row = {'participant_id': p_idx, 'diag': diag_lookup[p_idx]}
    for module in modules:
        coefs = coefficients[module][p_idx, 0].numpy()
        terms = candidate_terms[module]
        for t_idx, term in enumerate(terms):
            col = f"{module}_{term}"
            row[col] = float(coefs[t_idx])
            row[f"{col}_present"] = int(abs(coefs[t_idx]) > 1e-6)
    records.append(row)

df_coefs = pd.DataFrame(records)

# Logistic regression: coefficient presence ~ diagnosis group
reference = 'Control'
comparisons = ['Depression', 'Bipolar']
results = []

sindy_cols = [f"{m}_{t}" for m in modules for t in candidate_terms[m]]
for col in sindy_cols:
    pres_col = f"{col}_present"
    if pres_col not in df_coefs.columns:
        continue
    presence = df_coefs[pres_col].values
    if presence.std() == 0:
        continue  # skip if all same
    
    for comp in comparisons:
        mask = df_coefs['diag'].isin([reference, comp])
        sub = df_coefs[mask]
        y = sub[pres_col].values
        X = (sub['diag'] == reference).astype(int).values.reshape(-1, 1)
        
        if y.std() == 0 or len(sub) < 10:
            continue
        
        try:
            model = LogisticRegression(penalty=None, max_iter=1000)
            model.fit(X, y)
            beta = model.coef_[0, 0]
            
            # Standard error via Hessian approximation
            pred = model.predict_proba(X)[:, 1]
            W = np.diag(pred * (1 - pred))
            H = X.T @ W @ X
            se = np.sqrt(1.0 / max(H[0, 0], 1e-10))
            z = beta / se
            p_val = 2 * (1 - stats.norm.cdf(abs(z)))
            
            results.append({
                'coefficient': col,
                'comparison': f"{reference} vs {comp}",
                'beta': beta, 'se': se, 'p_value': p_val,
                'significant': p_val < 0.05,
                f'rate_{reference}': sub[sub['diag'] == reference][pres_col].mean(),
                f'rate_{comp}': sub[sub['diag'] == comp][pres_col].mean(),
            })
        except Exception:
            continue

df_results = pd.DataFrame(results)

# --- Forest plot ---
fig, axes = plt.subplots(1, len(comparisons), figsize=(7 * len(comparisons), max(6, len(sindy_cols) * 0.4)),
                          sharey=True)
if len(comparisons) == 1:
    axes = [axes]

for ax, comp in zip(axes, comparisons):
    sub = df_results[df_results['comparison'] == f"{reference} vs {comp}"].copy()
    if sub.empty:
        ax.set_title(f"{reference} vs {comp}\n(no testable coefficients)")
        continue
    
    sub = sub.sort_values('beta', key=abs, ascending=True).reset_index(drop=True)
    y_pos = np.arange(len(sub))
    
    colors = ['#FF4444' if p < 0.001 else '#FFA500' if p < 0.01 else '#FFD700' if p < 0.05 else '#999999'
              for p in sub['p_value']]
    
    for y, (_, row) in zip(y_pos, sub.iterrows()):
        lo, hi = row['beta'] - 1.96 * row['se'], row['beta'] + 1.96 * row['se']
        ax.plot([lo, hi], [y, y], color=colors[y], linewidth=2)
    
    ax.scatter(sub['beta'], y_pos, c=colors, s=50, zorder=5, edgecolors='black', linewidths=0.5)
    ax.axvline(0, color='black', linestyle='--', linewidth=0.8, alpha=0.7)
    ax.set_yticks(y_pos)
    ax.set_yticklabels(sub['coefficient'], fontsize=8)
    ax.set_xlabel('Effect size (beta)')
    ax.set_title(f'{reference} vs {comp}')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

plt.suptitle('Structural group differences: log-odds of coefficient presence', fontsize=12)
plt.tight_layout()
plt.show()

# Print significant findings
sig = df_results[df_results['significant']]
if len(sig) > 0:
    print(f"\nSignificant structural differences (p < 0.05):")
    for _, row in sig.iterrows():
        print(f"  {row['coefficient']:40s} {row['comparison']:25s} beta={row['beta']:+.3f}  p={row['p_value']:.4f}")
else:
    print("\nNo statistically significant structural differences found at p < 0.05")

## 5.5 Visualizing Individual Equations

Let's select structurally **distinctive** participants — those with maximally different equation structures — and visualize their discovered equations alongside their behavioral data.

In [ ]:
# Select structurally distinctive participants via greedy max-min Hamming distance
def select_distinctive_participants(estimator, n_select=3):
    """Greedy selection of participants with maximally different sparsity patterns."""
    coefficients = estimator.get_sindy_coefficients(aggregate=True)
    modules = estimator.get_modules()
    
    # Build binary sparsity vector per participant
    patterns = []
    for p in range(estimator.model.n_participants):
        pattern = []
        for m in modules:
            coefs = coefficients[m][p, 0].numpy()
            pattern.extend((np.abs(coefs) > 1e-6).astype(int).tolist())
        patterns.append(pattern)
    patterns = np.array(patterns)
    
    # Greedy max-min Hamming distance
    selected = [0]
    for _ in range(n_select - 1):
        best_idx, best_min_dist = -1, -1
        for i in range(len(patterns)):
            if i in selected:
                continue
            min_dist = min(np.sum(patterns[i] != patterns[s]) for s in selected)
            if min_dist > best_min_dist:
                best_idx, best_min_dist = i, min_dist
        selected.append(best_idx)
    return selected

selected_ids = select_distinctive_participants(estimator_clinical, n_select=4)
print(f"Selected participants: {selected_ids}")
print(f"Diagnoses: {[diag_lookup[pid] for pid in selected_ids]}")

# Show equations and action probabilities for each
estimator_clinical.model.eval()
estimator_clinical.model.use_sindy = True

for pid in selected_ids:
    print(f"\n{'='*60}")
    print(f"Participant {pid} ({diag_lookup[pid]})")
    print(f"{'='*60}")
    estimator_clinical.print_spice_model(participant_id=pid)
    
    # Get action probabilities for one session of this participant
    session_mask = dataset_clinical.xs[:, 0, 0, -1].int() == pid
    session_indices = torch.where(session_mask)[0]
    if len(session_indices) == 0:
        continue
    
    sid = session_indices[0].item()
    xs_session = dataset_clinical.xs[sid:sid+1]
    
    with torch.no_grad():
        logits_out, _ = estimator_clinical.model(xs_session)
    
    logits_np = logits_out[0, :, 0, :].numpy()
    probs_np = np.exp(logits_np) / np.exp(logits_np).sum(axis=-1, keepdims=True)
    
    # Get actual choices for this session
    choices = dataset_clinical.ys[sid, :, 0, :].numpy()
    valid = ~np.isnan(choices[:, 0])
    n_valid = valid.sum()
    
    fig, axes = plt.subplots(2, 1, figsize=(10, 4), sharex=True)
    
    # Choices
    ax = axes[0]
    for t in range(n_valid):
        choice = np.argmax(choices[t])
        color = 'C0' if choice == 0 else 'C1'
        ax.scatter(t, choice, color=color, s=8, alpha=0.5)
    ax.set_ylabel('Choice')
    ax.set_yticks([0, 1])
    ax.set_title(f'Participant {pid} ({diag_lookup[pid]})')
    
    # Action probabilities
    ax = axes[1]
    ax.plot(range(n_valid), probs_np[:n_valid, 0], 'C0', alpha=0.7, label='P(arm 0)')
    ax.plot(range(n_valid), probs_np[:n_valid, 1], 'C1', alpha=0.7, label='P(arm 1)')
    ax.set_xlabel('Trial')
    ax.set_ylabel('P(action)')
    ax.set_ylim(-0.05, 1.05)
    ax.legend(loc='upper right', fontsize=8)
    
    plt.tight_layout()
    plt.show()

## 5.6 Presence Rates by Diagnostic Group

In [ ]:
# Grouped bar chart: presence rates per coefficient by diagnostic group
group_order = ['Control', 'Depression', 'Bipolar']
group_colors = {'Control': '#4CAF50', 'Depression': '#F44336', 'Bipolar': '#2196F3'}

# Compute presence rates per group
presence_data = []
for col in sindy_cols:
    pres_col = f"{col}_present"
    if pres_col not in df_coefs.columns:
        continue
    for grp in group_order:
        grp_data = df_coefs[df_coefs['diag'] == grp][pres_col]
        if len(grp_data) > 0:
            presence_data.append({
                'Coefficient': col,
                'Group': grp,
                'Presence Rate': grp_data.mean(),
            })

pres_df = pd.DataFrame(presence_data)
pivot = pres_df.pivot(index='Coefficient', columns='Group', values='Presence Rate')
pivot = pivot[group_order]

# Only show coefficients with some variation
varying = pivot[(pivot.max(axis=1) - pivot.min(axis=1)) > 0.05]
if len(varying) == 0:
    varying = pivot.head(10)

fig, ax = plt.subplots(figsize=(max(10, len(varying) * 1.2), 5))
varying.plot(kind='bar', ax=ax, color=[group_colors[g] for g in group_order], width=0.8)
ax.set_ylabel('Presence Rate')
ax.set_title('Coefficient Presence Rates by Diagnostic Group')
ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right', fontsize=8)
ax.legend(title='Diagnosis')
ax.set_ylim(0, 1.05)
plt.tight_layout()
plt.show()

---
# Summary

### What we covered:

| Concept | Key takeaway |
|---------|--------------|
| **Data pipeline** | `DataFrame` → `csv_to_dataset()` → `SpiceDataset` |
| **SpiceConfig** | Defines modules, their inputs, and memory states |
| **BaseModel** | `setup_modules_from_config()` + `call_module()` in a trial loop |
| **action_mask** | Controls which items get updated (chosen vs unchosen vs all) |
| **Ensemble** | Multiple RNNs trained in parallel for robust pruning |
| **Equations** | Discovered in logit space (scaled by β) |
| **Individual differences** | Per-participant SINDy coefficients reveal structural variations |
| **Additional inputs** | Pass metadata through dataset without affecting SINDy library |
| **Feature derivation** | Precompute features in `forward()` for polynomial-amenable architecture |
| **Real data** | Applied SPICE to clinical data with diagnostic group comparisons |

### Key design principles:

1. The model architecture defines the **hypothesis space**. SPICE searches within it, using sparsification to find the simplest equations that explain the data.
2. **Derived features** (prediction error, running averages, etc.) computed in the forward pass make the model more polynomial-amenable.
3. **Structural individual differences** — not just different parameter values, but different equation structures — can reveal meaningful clinical distinctions.

### Discussion questions:
1. What structural differences did SPICE find between diagnostic groups?
2. How might derived features (prediction error, reward history) change the discovered equations?
3. What are the limitations of structural equation discovery for clinical diagnosis?
4. How does the choice of model architecture (which modules, which inputs) affect what SPICE can discover?

### Next steps:
- Experiment with additional derived features in `ClinicalRW.forward()`
- Try different `polynomial_degree` settings per module
- Explore the [SPICE documentation](https://whyhardt.github.io/SPICE/) for advanced features